# Visual Anomaly Detection - test end-to-end

Ten notebook uruchamia caly prototyp: generuje dane syntetyczne, tworzy obrazy wykresow, trenuje baseline numeryczny, opcjonalnie trenuje model wizualny CNN i porownuje wyniki.

Domyslnie notebook uzywa malego zbioru `dataset_notebook_demo`, zeby test byl szybki. Pelny eksperyment mozna uruchomic przez zwiekszenie `N_SERIES`, `SERIES_LENGTH`, `VISUAL_EPOCHS` i ustawienie katalogu `dataset`.

## 1. Konfiguracja sciezek i zaleznosci

In [ ]:
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data_generator.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_DIR = PROJECT_ROOT / "dataset_notebook_demo"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts_notebook_demo"
NUMERICAL_MODEL_PATH = ARTIFACTS_DIR / "numerical_baseline.pkl"
VISUAL_MODEL_PATH = ARTIFACTS_DIR / "visual_detector.pt"

RANDOM_STATE = 42
N_SERIES = 60
SERIES_LENGTH = 320
WINDOW_SIZE = 80
STRIDE = 40
VISUAL_EPOCHS = 2

HAS_SKLEARN = importlib.util.find_spec("sklearn") is not None
HAS_TORCH = importlib.util.find_spec("torch") is not None

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset dir: {DATASET_DIR}")
print(f"Artifacts dir: {ARTIFACTS_DIR}")
print(f"scikit-learn available: {HAS_SKLEARN}")
print(f"PyTorch available: {HAS_TORCH}")

## 2. Generowanie danych i obrazow wykresow

Generator tworzy szereg czasowy jako trend + sezonowosc + szum, a potem wstrzykuje trzy typy anomalii: pojedyncze skoki, anomalie kontekstowe i zmiany wariancji. Kazde okno czasowe dostaje etykiete `normal` albo `anomaly`. Na koncu generator balansuje klasy osobno w `train`, `val` i `test`, wiec kazdy split ma 50% wykresow normalnych i 50% wykresow z anomalia.

In [ ]:
from data_generator import GeneratorConfig, build_dataset

generator_config = GeneratorConfig(
    output_dir=DATASET_DIR,
    n_series=N_SERIES,
    series_length=SERIES_LENGTH,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    anomaly_series_fraction=0.7,
    random_state=RANDOM_STATE,
    plot_backend="auto",
    overwrite=True,
)

numeric_df, metadata_df = build_dataset(generator_config)
print(f"Time points: {len(numeric_df):,}")
print(f"Chart windows: {len(metadata_df):,}")
display(metadata_df.groupby(["split", "label_name"]).size().unstack(fill_value=0))

## 3. Podglad przykladowych wykresow

In [ ]:
from IPython.display import display
from PIL import Image

sample_rows = metadata_df.groupby("label_name", group_keys=False).head(3)
for row in sample_rows.itertuples(index=False):
    print(f"{row.split} / {row.label_name} / anomalies in window: {row.anomaly_count}")
    image = Image.open(DATASET_DIR / row.image_path).convert("RGB")
    image.thumbnail((420, 240))
    display(image)

## 4. Trening baseline numerycznego

Baseline nie patrzy na obraz. Dla kazdego okna wylicza cechy statystyczne, np. srednia, odchylenie, zakres, IQR, nachylenie trendu, odchylenia reszt i skoki miedzy punktami. Jesli jest dostepny scikit-learn, uzywa Random Forest. W przeciwnym razie uzywa lekkiego fallbacku `robust_threshold`.

In [ ]:
from numerical_baseline import NumericalAnomalyBaseline, load_window_dataset
from evaluation import binary_metrics

features, y_all, all_meta = load_window_dataset(DATASET_DIR)
train_mask = all_meta["split"].eq("train").to_numpy()
test_mask = all_meta["split"].eq("test").to_numpy()

numerical_model_type = "random_forest" if HAS_SKLEARN else "robust_threshold"
numerical_model = NumericalAnomalyBaseline(
    model_type=numerical_model_type,
    random_state=RANDOM_STATE,
)
numerical_model.fit(features.loc[train_mask], y_all[train_mask])
numerical_model.save(NUMERICAL_MODEL_PATH)

y_num = y_all[test_mask]
num_score = numerical_model.predict_proba(features.loc[test_mask])
num_meta = all_meta.loc[test_mask].reset_index(drop=True)
num_metrics = binary_metrics(y_num, num_score)

print(f"Numerical model: {numerical_model_type}")
print(f"Saved to: {NUMERICAL_MODEL_PATH}")
display(num_metrics)

## 5. Trening modelu wizualnego CNN

Ten krok wymaga PyTorch. Model dostaje tylko obraz PNG wykresu i uczy sie klasyfikacji `normal` vs `anomaly`. Jesli `torch` nie jest zainstalowany, komorka pominie trening i pokaze komende instalacji.

In [ ]:
if HAS_TORCH:
    from visual_anomaly_detector import VisualTrainingConfig, train_visual_detector

    visual_config = VisualTrainingConfig(
        image_size=128,
        batch_size=16,
        epochs=VISUAL_EPOCHS,
        learning_rate=1e-3,
        device="auto",
        random_state=RANDOM_STATE,
    )
    visual_detector = train_visual_detector(
        dataset_dir=DATASET_DIR,
        model_out=VISUAL_MODEL_PATH,
        config=visual_config,
    )
    y_vis, vis_score, vis_meta = visual_detector.predict_dataset_split(DATASET_DIR, split="test")
    vis_metrics = binary_metrics(y_vis, vis_score)
    print(f"Saved to: {VISUAL_MODEL_PATH}")
    display(vis_metrics)
else:
    y_vis, vis_score, vis_meta, vis_metrics = None, None, None, None
    print("PyTorch is not installed, so visual CNN training was skipped.")
    print("Install project dependencies first, for example:")
    print("python -m pip install -r requirements.txt")

## 6. Porownanie metod i przypadki rozbieznosci

In [ ]:
from evaluation import build_report, save_case_sheets

if HAS_TORCH:
    report_df, comparison_df = build_report(
        numerical_result=(y_num, num_score, num_meta),
        visual_result=(y_vis, vis_score, vis_meta),
    )
    report_path = ARTIFACTS_DIR / "evaluation_report.csv"
    comparison_path = ARTIFACTS_DIR / "prediction_comparison.csv"
    report_df.to_csv(report_path, index=False)
    comparison_df.to_csv(comparison_path, index=False)
    save_case_sheets(DATASET_DIR, comparison_df, ARTIFACTS_DIR, max_cases=9)

    print(f"Report saved to: {report_path}")
    print(f"Comparison saved to: {comparison_path}")
    display(report_df)
else:
    print("Full comparison needs the visual model. Run this notebook again after installing PyTorch.")

In [ ]:
if HAS_TORCH:
    for file_name in ["visual_better_than_numerical.png", "numerical_better_than_visual.png"]:
        image_path = ARTIFACTS_DIR / file_name
        print(image_path.name)
        display(Image.open(image_path).convert("RGB"))

## 7. Jak czytac wynik

- `accuracy` mowi, jaki procent okien zostal poprawnie sklasyfikowany.
- `precision` mowi, ile alarmow faktycznie bylo anomaliami.
- `recall` mowi, ile wszystkich anomalii model znalazl.
- `f1_score` laczy precision i recall, dlatego jest dobry przy nierownych klasach.
- `roc_auc` ocenia jak dobrze wynik punktowy rozdziela normalne i anomalne okna niezaleznie od progu 0.5.

W praktyce model numeryczny czesto wygrywa na oczywistych skokach wartosci. Model wizualny moze byc ciekawszy tam, gdzie wazny jest ksztalt wykresu, zmiana rytmu, lokalna nienaturalnosc albo artefakty widoczne na obrazie, ale slabsze w prostych cechach statystycznych.